# MODULE 5 — LLM & VLM pour la GeoAI
## Cours Magistral Interactif

> **Cours** : Analyse Spatiale avec Machine Learning (ASML)  
> **Institut** : 2iE

| OA | Objectif | Section |
|----|---------|---------|
| OA1 | SAM — segmentation d'images satellites sans supervision | §2 |
| OA2 | VLM — description automatique de cartes | §3 |
| OA3 | Agent LangChain pour requêtes géospatiales | §4 |
| OA4 | RAG — interroger les rapports CH en langage naturel | §5 |
| OA5 | Pipeline GeoAI complet : données → bulletin | §6 |
| OA6 | Limites et éthique des LLM pour la décision humanitaire | §7 |

> Les **exercices** sont dans `M5_TP_Enonce.ipynb` — agent alerte hydraulique + rapport VLM.

In [ ]:
# Installation — décommenter selon les besoins
# !pip install langchain langchain-community langchain-openai chromadb
#             sentence-transformers transformers torch pillow --quiet
# !pip install segment-anything  # SAM (Meta AI)
# Pour Mistral local : installer Ollama (ollama.ai) puis : ollama pull mistral

In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd
from shapely.geometry import Point

DF_JSON      = '[{"province": "Oudalan", "region": "Sahel", "lat": 14.48, "lon": -0.48, "ipc_phase": 4, "fapar_mean": 0.06, "dist_route_km": 245, "acces_eau_pct": 28}, {"province": "Séno", "region": "Sahel", "lat": 14.1, "lon": -0.1, "ipc_phase": 4, "fapar_mean": 0.06, "dist_route_km": 198, "acces_eau_pct": 35}, {"province": "Soum", "region": "Sahel", "lat": 13.98, "lon": -1.55, "ipc_phase": 4, "fapar_mean": 0.07, "dist_route_km": 220, "acces_eau_pct": 42}, {"province": "Yagha", "region": "Sahel", "lat": 13.35, "lon": 0.75, "ipc_phase": 3, "fapar_mean": 0.09, "dist_route_km": 185, "acces_eau_pct": 30}, {"province": "Loroum", "region": "Nord", "lat": 13.7, "lon": -2.1, "ipc_phase": 3, "fapar_mean": 0.11, "dist_route_km": 120, "acces_eau_pct": 45}, {"province": "Yatenga", "region": "Nord", "lat": 13.55, "lon": -2.35, "ipc_phase": 3, "fapar_mean": 0.13, "dist_route_km": 85, "acces_eau_pct": 52}, {"province": "Passoré", "region": "Nord", "lat": 12.89, "lon": -2.2, "ipc_phase": 3, "fapar_mean": 0.15, "dist_route_km": 65, "acces_eau_pct": 55}, {"province": "Sanmatenga", "region": "Centre-Nord", "lat": 13.05, "lon": -1.2, "ipc_phase": 3, "fapar_mean": 0.16, "dist_route_km": 72, "acces_eau_pct": 58}, {"province": "Namentenga", "region": "Centre-Nord", "lat": 13.1, "lon": -0.5, "ipc_phase": 3, "fapar_mean": 0.18, "dist_route_km": 95, "acces_eau_pct": 54}, {"province": "Bam", "region": "Centre-Nord", "lat": 13.45, "lon": -1.4, "ipc_phase": 2, "fapar_mean": 0.19, "dist_route_km": 80, "acces_eau_pct": 60}, {"province": "Kadiogo", "region": "Centre", "lat": 12.36, "lon": -1.53, "ipc_phase": 2, "fapar_mean": 0.25, "dist_route_km": 12, "acces_eau_pct": 88}, {"province": "Bazega", "region": "Centre-Sud", "lat": 12.1, "lon": -1.4, "ipc_phase": 2, "fapar_mean": 0.28, "dist_route_km": 45, "acces_eau_pct": 65}, {"province": "Houet", "region": "Hauts-Bassins", "lat": 11.18, "lon": -4.3, "ipc_phase": 2, "fapar_mean": 0.38, "dist_route_km": 22, "acces_eau_pct": 78}, {"province": "Comoé", "region": "Cascades", "lat": 10.6, "lon": -4.65, "ipc_phase": 1, "fapar_mean": 0.44, "dist_route_km": 85, "acces_eau_pct": 82}, {"province": "Poni", "region": "Sud-Ouest", "lat": 10.3, "lon": -3.3, "ipc_phase": 2, "fapar_mean": 0.48, "dist_route_km": 95, "acces_eau_pct": 79}]'
RAPPORTS_JSON = '[{"source": "CH_BF_2024_T1.txt", "contenu": "La province de Séno enregistre une situation alimentaire critique en Phase 4. La pluviométrie cumulée 2023 est inférieure de 35% à la moyenne décennale. Les marchés de Dori présentent des prix du mil supérieurs de 45% à la normale. Un mouvement de 12 000 déplacés internes vers la province voisine de Soum est signalé."}, {"source": "CH_BF_2024_T1.txt", "contenu": "Oudalan : Phase 4 confirmée. Les ménages en stratégie de survie de crise représentent 68%. Accès humanitaire limité sur 3 communes (insécurité). Besoins en assistance : 145 000 personnes."}, {"source": "FEWS_BF_mars2024.txt", "contenu": "Les Hauts-Bassins (Houet, Comoé) maintiennent une situation en Phase 2 grâce aux bonnes récoltes 2023. Excédents céréaliers estimés à +18% vs moyenne. Les flux commerciaux vers le Nord restent bloqués par l\'insécurité."}, {"source": "FEWS_BF_mars2024.txt", "contenu": "Le Centre-Nord (Sanmatenga, Namentenga) bascule vers Phase 3. FAPAR juillet 2023 = 0.14, en baisse de 12% vs 2022. Récolte sorgho déficitaire de 22%. Marchés approvisionnés mais prix élevés."}, {"source": "ANAM_BF_2024.txt", "contenu": "Prévisions climatiques mars-mai 2024 : déficit pluviométrique probable dans le Sahel BF. Probabilité de sécheresse modérée à sévère : 70% pour les provinces de Soum, Séno, Oudalan. Risque de détérioration IPC pour 8 provinces du Nord d\'ici juin 2024."}]'

df = pd.read_json(DF_JSON)
rapports = json.loads(RAPPORTS_JSON)

print(f'Données : {len(df)} provinces BF')
print(f'Rapports simulés : {len(rapports)} documents')
print(df[['province','ipc_phase','fapar_mean','dist_route_km']].head())

---
# §2 — SAM : Segmentation d'Images Satellites

> 🎯 **OA1** — SAM pour délimiter des zones d'intérêt sans entraînement.

SAM prend une image et un point/boîte en entrée → retourne un masque précis.
Sans SAM : délimiter manuellement 45 provinces = plusieurs semaines.
Avec SAM : quelques minutes, en Python.

> ⚠️ SAM nécessite un GPU et des poids (~2.5 Go). Ce notebook simule son comportement.
> En TP, vous l'exécuterez sur Colab GPU avec de vraies images Sentinel-2.

In [ ]:
# OA1 — Simulation de la sortie SAM sur une image synthétique
# (En production : remplacer par de vraies images Sentinel-2 + SAM réel)

np.random.seed(42)
IMAGE_SIZE = 256

def simulate_sentinel2_image(lat_center, ipc_phase):
    '''Simule un patch Sentinel-2 (RGB approximatif) selon la zone BF.'''
    # NDVI lié à la latitude (gradient N-S BF)
    ndvi_base = np.clip(0.55 - (lat_center - 9.5) * 0.06, 0.05, 0.60)
    # Bandes RGB simplifiées
    r = np.random.uniform(0.08, 0.18, (IMAGE_SIZE, IMAGE_SIZE))
    g = np.random.uniform(0.06, 0.15, (IMAGE_SIZE, IMAGE_SIZE))
    b = np.random.uniform(0.04, 0.10, (IMAGE_SIZE, IMAGE_SIZE))
    # Ajouter des zones agricoles (rectangles plus verts)
    n_fields = max(1, int(ndvi_base * 20))
    for _ in range(n_fields):
        x, y = np.random.randint(20, IMAGE_SIZE-40, 2)
        w, h = np.random.randint(15, 40, 2)
        boost = ndvi_base * 0.6
        g[y:y+h, x:x+w] += boost
        b[y:y+h, x:x+w] += boost * 0.3
    return np.stack([np.clip(r,0,1), np.clip(g,0,1), np.clip(b,0,1)], axis=-1)

def simulate_sam_mask(image, point_x, point_y, radius=30):
    '''Simule un masque SAM circulaire autour d'un point prompt.'''
    H, W = image.shape[:2]
    Y, X = np.ogrid[:H, :W]
    dist = np.sqrt((X - point_x)**2 + (Y - point_y)**2)
    # Masque avec bords irréguliers (simulation réaliste)
    noise = np.random.normal(0, radius*0.15, (H, W))
    mask = (dist + noise) < radius
    score = float(np.clip(0.85 + np.random.normal(0, 0.05), 0.70, 0.98))
    return mask, score

# Démonstration sur 3 provinces contrastées
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
exemples = [
    ('Séno',    14.10, 4, 'IPC 4 — Sahel aride'),
    ('Kadiogo', 12.36, 2, 'IPC 2 — Centre BF'),
    ('Comoé',   10.60, 1, 'IPC 1 — Zone humide'),
]

for j, (prov, lat, ipc, label) in enumerate(exemples):
    image = simulate_sentinel2_image(lat, ipc)
    px, py = IMAGE_SIZE//2 + np.random.randint(-20,20,2)
    mask, score = simulate_sam_mask(image, px, py)

    # Image originale
    axes[0,j].imshow(image)
    axes[0,j].plot(px, py, 'r+', markersize=15, markeredgewidth=2,
                   label='Point prompt')
    axes[0,j].set_title(f'{prov} — {label}', fontsize=10, fontweight='bold')
    axes[0,j].legend(fontsize=8); axes[0,j].axis('off')

    # Masque SAM
    overlay = image.copy()
    overlay[mask] = overlay[mask] * 0.5 + np.array([1,0.6,0]) * 0.5
    axes[1,j].imshow(overlay)
    surf = mask.sum() * 100  # 10m/pixel → 100m² par pixel
    axes[1,j].set_title(f'Masque SAM — score={score:.2f}\n{surf/1e4:.1f} ha détectés',
                         fontsize=9, fontweight='bold')
    axes[1,j].axis('off')

plt.suptitle('SAM — Segmentation automatique de zones agricoles Sentinel-2 BF',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.savefig('M5_cm_sam_demo.png', dpi=120); plt.show()
print('↔️ En production : SamPredictor.predict(point_coords, point_labels)')

---
# §3 — VLM : Description Automatique de Cartes

> 🎯 **OA2** — Utiliser un VLM pour décrire une carte en langage naturel.

On produit d'abord la carte IPC (comme en M3), puis on la décrit avec un VLM.
GPT-4V (OpenAI) ou LLaVA (open source Colab GPU) font ce travail.

In [ ]:
# OA2 — Étape 1 : Générer la carte IPC prédite (output M3)

gdf = gpd.GeoDataFrame(
    df,
    geometry=[Point(r.lon, r.lat) for _, r in df.iterrows()],
    crs='EPSG:4326'
)

palette = {1:'#27ae60', 2:'#f1c40f', 3:'#e67e22', 4:'#e74c3c'}
fig, ax = plt.subplots(figsize=(10, 8))

for ipc, grp in gdf.groupby('ipc_phase'):
    grp.plot(ax=ax, color=palette.get(ipc,'#95a5a6'),
             markersize=200, marker='o', label=f'Phase {ipc}', alpha=0.85)

for _, row in gdf.iterrows():
    ax.annotate(row['province'][:6],
                xy=(row.geometry.x, row.geometry.y),
                xytext=(5, 5), textcoords='offset points',
                fontsize=7, color='#2c3e50')

leg = [mpatches.Patch(color=v, label=f'Phase {k}') for k,v in palette.items()]
ax.legend(handles=leg, loc='lower left', fontsize=10)
ax.set_title('Phases IPC prédites — Burkina Faso (Modèle RF M3)',
             fontsize=12, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('M5_cm_carte_ipc.png', dpi=150, bbox_inches='tight')
plt.show()
print('Carte sauvegardée : M5_cm_carte_ipc.png → prête pour analyse VLM')

In [ ]:
# OA2 — Étape 2 : Simulation de la réponse VLM
# (En production : appeler GPT-4V ou LLaVA avec la carte en base64)

def simulate_vlm_description(df_pred):
    '''Simule la sortie d\'un VLM décrivant la carte IPC.'''
    p4 = df_pred[df_pred['ipc_phase']>=4]['province'].tolist()
    p3 = df_pred[df_pred['ipc_phase']==3]['province'].tolist()
    p1 = df_pred[df_pred['ipc_phase']<=1]['province'].tolist()
    return (
        f'La carte présente une situation alimentaire préoccupante dans le Nord du Burkina Faso. '
        f'Les provinces du Sahel ({', '.join(p4)}) apparaissent en rouge foncé, '
        f'indiquant une Phase 4 (Urgence). '
        f'Le Centre-Nord ({', '.join(p3[:3])}) est en orange (Phase 3). '
        f'Le Sud-Ouest ({', '.join(p1)}) présente une végétation dense (vert), '
        f'confirmant une situation alimentaire meilleure (Phase 1-2). '
        f'Un gradient Nord-Sud marqué est visible, cohérent avec le gradient pluviométrique.'
    )

description_vlm = simulate_vlm_description(df)
print('=== Description VLM de la carte IPC BF ===')
print(description_vlm)
print()
print('↔️ En production :')
print('  img_b64 = encode_image_base64("M5_cm_carte_ipc.png")')
print('  desc = analyze_ipc_map_gpt4v(img_b64, api_key, context=...)')
print('  # ou : describe_map_llava("M5_cm_carte_ipc.png", question)')

---
# §4 — Agent LangChain pour Requêtes Géospatiales

> 🎯 **OA3** — Construire un agent qui répond en langage naturel.

Un agent LLM possède des **outils** (fonctions Python) qu'il choisit d'appeler
selon la requête. Il raisonne (ReAct), choisit un outil, l'exécute, observe,
et continue jusqu'à la réponse finale.

In [ ]:
# OA3 — Agent géospatial simulé (sans API payante)
# Simule le comportement d'un agent LangChain complet

class GeoSpatialAgentSimule:
    '''Agent géospatial simulé — même interface qu\'un vrai agent LangChain.'''

    def __init__(self, df):
        self.df = df
        self.historique = []

    # ─── Outils de l'agent ─────────────────────────────────────────
    def _requete_ipc(self, query):
        q = query.lower()
        if 'phase 4' in q or 'urgence' in q:
            p = self.df[self.df['ipc_phase']>=4]['province'].tolist()
            return f'{len(p)} provinces Phase 4+ : {', '.join(p)}'
        elif 'enclavée' in q or 'route' in q and '150' in q:
            enc = self.df[self.df['dist_route_km']>150]
            return enc[['province','dist_route_km']].to_string(index=False)
        elif 'fapar' in q:
            low = self.df[self.df['fapar_mean']<0.10]['province'].tolist()
            return f'Provinces FAPAR < 0.10 : {', '.join(low)}'
        return self.df[['province','ipc_phase','fapar_mean']].to_string(index=False)

    def _score_risque(self, province):
        if province not in self.df['province'].values:
            return f'{province} non trouvée.'
        r = self.df[self.df['province']==province].iloc[0]
        s = r['ipc_phase']/5*0.4 + (1-r['fapar_mean'])*0.3 + r['dist_route_km']/300*0.3
        s = round(min(s,1.0), 2)
        niv = 'Critique' if s>0.7 else ('Élevé' if s>0.5 else 'Modéré')
        return f'{province} : score={s} — {niv}'

    def _liste_provinces_prioritaires(self, _):
        top5 = self.df.nlargest(5,'ipc_phase')[['province','ipc_phase','dist_route_km']]
        return top5.to_string(index=False)

    # ─── Moteur de raisonnement simulé ─────────────────────────────
    def run(self, question):
        self.historique.append(f'Q: {question}')
        print(f'[Agent] Analyse : "{question}"')

        # Sélection d'outil basée sur mots-clés (simule le LLM ReAct)
        q = question.lower()
        if 'score' in q or 'risque' in q:
            # Extraire le nom de province
            for _, row in self.df.iterrows():
                if row['province'].lower() in q:
                    outil, args = 'ScoreRisque', row['province']
                    break
            else:
                outil, args = 'ScoreRisque', self.df.nlargest(1,'ipc_phase').iloc[0]['province']
        elif 'prioritaire' in q or 'top' in q:
            outil, args = 'ListePrioritaires', ''
        else:
            outil, args = 'RequêteIPCBF', question

        print(f'[Agent] Outil choisi : {outil}({args!r})')

        if outil == 'ScoreRisque':
            result = self._score_risque(args)
        elif outil == 'ListePrioritaires':
            result = self._liste_provinces_prioritaires(args)
        else:
            result = self._requete_ipc(args)

        print(f'[Agent] Résultat outil : {result}')

        # Synthèse finale (simule la réponse LLM)
        reponse = f'Selon les données géospatiales BF : {result}'
        self.historique.append(f'R: {reponse}')
        return reponse


agent = GeoSpatialAgentSimule(df)

questions = [
    'Quelles provinces sont en Phase 4 (urgence) ?',
    'Quelles provinces sont enclavées avec dist_route > 150 km ?',
    'Donne-moi le score de risque de Séno.',
    'Montre-moi les 5 provinces les plus prioritaires.',
]

for q in questions:
    print('─' * 60)
    rep = agent.run(q)
    print(f'Réponse finale : {rep}')
    print()

---
# §5 — RAG : Interroger les Rapports CH en Langage Naturel

> 🎯 **OA4** — RAG pour ancrer les réponses LLM dans les vrais documents.

**Problème des LLM sans RAG** : ils inventent des chiffres IPC plausibles mais faux.
**Solution RAG** : indexer les vrais rapports CH → recherche sémantique → réponse ancrée.

In [ ]:
# OA4 — RAG simulé avec recherche par mots-clés
# (En production : ChromaDB + sentence-transformers + LLM)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Indexation TF-IDF (simule les embeddings vectoriels du vrai RAG)
corpus = [r['contenu'] for r in rapports]
sources = [r['source'] for r in rapports]

vectorizer = TfidfVectorizer(ngram_range=(1,2))
tfidf_matrix = vectorizer.fit_transform(corpus)
print(f'Index RAG : {len(corpus)} documents, {tfidf_matrix.shape[1]} termes')

def rag_query(question, k=2):
    '''Recherche les k passages les plus pertinents pour une question.'''
    q_vec = vectorizer.transform([question])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_k = scores.argsort()[::-1][:k]
    return [(corpus[i], sources[i], scores[i]) for i in top_k if scores[i] > 0.01]

def rag_answer(question, k=3):
    '''Génère une réponse ancrée dans les documents.'''
    passages = rag_query(question, k)
    if not passages:
        return 'Aucun document pertinent trouvé. Question hors du corpus disponible.'

    context = '\n'.join([f'[{src}] {txt}' for txt, src, _ in passages])

    # Simulation de la réponse LLM avec le contexte RAG
    # En production : llm.predict(f'Question: {question}\nContexte: {context}\nRéponse:')
    reponse = (
        f'Sur la base des rapports officiels (CH + FEWS NET) :\n'
        f'{context[:500]}\n\n'
        f'→ Résumé : {passages[0][0][:150]}...'
    )
    return reponse, [(s, sc) for _, s, sc in passages]

# Tests
questions_rag = [
    'Quelles provinces sont en urgence alimentaire Phase 4 ?',
    'Quelle est la situation dans les Hauts-Bassins ?',
    'Quelles sont les prévisions climatiques pour le Sahel BF ?',
]

for q in questions_rag:
    print(f'Q: {q}')
    res, srcs = rag_answer(q)
    print(f'Sources : {[s for s,_ in srcs]}')
    print(f'Extrait : {res[:200]}...')
    print()

---
# §6 — Pipeline GeoAI Complet : Données → Bulletin

> 🎯 **OA5** — Assembler tous les composants en un pipeline bout-en-bout.

In [ ]:
# OA5 — Pipeline complet : RF M3 + SAM + VLM + RAG → Bulletin humanitaire

def generate_bulletin_humanitaire(df_pred, rapports_simules):
    '''
    Pipeline GeoAI complet :
    1. Extraire stats depuis prédictions M3
    2. Description VLM de la carte (simulée)
    3. Contexte RAG depuis rapports CH
    4. Génération bulletin (simulée)
    '''
    # 1. Stats prédictions M3
    urgences = df_pred[df_pred['ipc_phase']>=4]
    crises   = df_pred[df_pred['ipc_phase']==3]
    stats = (
        f'{len(urgences)} provinces Phase 4 (Urgence), '
        f'{len(crises)} Phase 3 (Crise). '
        f'Provinces urgence : {', '.join(urgences.province.tolist())}. '
        f'FAPAR moyen zones urgence : {urgences.fapar_mean.mean():.3f}.'
    )

    # 2. Description VLM (simulée — remplacer par appel GPT-4V ou LLaVA)
    vlm_desc = simulate_vlm_description(df_pred)

    # 3. RAG
    ctx_passages = rag_query('urgence alimentaire phase 4 sécheresse prévisions', k=3)
    context_rag = '\n'.join([t[:200] for t,s,_ in ctx_passages])

    # 4. Bulletin généré (simulation LLM — remplacer par llm.predict(prompt))
    bulletin = f'''
BULLETIN D'ALERTE PRÉCOCE — BURKINA FASO
Généré par le système GeoAI ASML | {pd.Timestamp.now().strftime('%d %B %Y')}
═══════════════════════════════════════

SITUATION GÉNÉRALE
Le modèle ML (Random Forest, κ spatial CV = 0.45) identifie une situation
alimentaire critique dans le Nord-Est du pays.

DONNÉES MODÈLE : {stats}

ANALYSE VISUELLE (VLM) : {vlm_desc[:200]}

CONTEXTE DOCUMENTAIRE (CH/FEWS NET) :
{context_rag[:300]}

RECOMMANDATIONS PRIORITAIRES :
1. Pré-positionner des stocks d'urgence dans les provinces Phase 4
2. Activer les mécanismes de réponse rapide OCHA pour Séno et Oudalan
3. Surveiller les flux de déplacés entre Sahel et Centre-Nord

⚠️ CE BULLETIN EST GÉNÉRÉ AUTOMATIQUEMENT.
Il doit être validé par un analyste humanitaire avant toute décision.
═══════════════════════════════════════
'''
    return bulletin

bulletin = generate_bulletin_humanitaire(df, rapports)
print(bulletin)

---
# §7 — Limites et Éthique

> 🎯 **OA6** — Identifier les risques et conditions d'usage responsable.

| Risque | Mitigation |
|--------|-----------|
| **Hallucination** — LLM invente des chiffres IPC | RAG obligatoire + citations |
| **Biais géographique** — peu de données africaines | Validation par expert terrain |
| **Confidentialité** — données CH sensibles envoyées à OpenAI | LLM local (Mistral, LLaMA) |
| **Knowledge cutoff** — LLM ignore les rapports récents | Base RAG mise à jour |
| **Confiance excessive** — utilisateur croit sans vérifier | Formation + affichage sources |

> ⚠️ **Règle absolue** : un bulletin LLM n'est qu'un **premier draft**.
> Un expert humain signe et assume la responsabilité de la décision.

In [ ]:
# OA6 — Démonstration : LLM sans RAG vs avec RAG

# Simulation LLM sans RAG : peut halluciner
def llm_sans_rag(question):
    '''Simule un LLM qui répond sans accès aux vrais documents (risque hallucination).'''
    return (
        f'[LLM sans RAG] La province de Dori (Sahel) présente une situation '
        f'alimentaire critique avec 78% des ménages en Phase 4, selon le rapport '
        f'CH de mars 2024. Les prix du mil sont supérieurs de 52% à la normale. '
        f'[ATTENTION : ces chiffres sont INVENTÉS par le LLM — non vérifiables]'
    )

# Avec RAG : ancré dans les vrais documents
def llm_avec_rag(question):
    passages = rag_query(question, k=2)
    if not passages:
        return 'Information non trouvée dans les rapports disponibles.'
    extrait = passages[0][0]
    source  = passages[0][1]
    return (
        f'[LLM avec RAG] Selon {source} : "{extrait[:200]}..."\n'
        f'Source vérifiable : {source}'
    )

q = 'Situation alimentaire Séno Oudalan Phase 4'
print('=== SANS RAG (risque hallucination) ===')
print(llm_sans_rag(q))
print()
print('=== AVEC RAG (ancré dans les documents) ===')
print(llm_avec_rag(q))
print()
print('→ Toujours utiliser RAG pour les décisions humanitaires.')
print('→ Afficher les sources à l\'utilisateur final.')